# LoRA Fine-Tuning for Sequence Classification
This notebook follows the checkpoints in the README to fine-tune BERT using LoRA.

In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
import torch 
from peft import LoraConfig, get_peft_model
import evaluate 

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

c:\Users\VICTUS\OneDrive\Desktop\Generative AI\genAI-playground\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [3]:
# Load Dataset (e.g., dair-ai/emotion)
dataset = load_dataset("dair-ai/emotion")
emotion_labels = dataset["train"].features["label"].names
print("Labels:", emotion_labels)

dataset["train"] = dataset["train"].shuffle(seed=42).select(range(3000))
dataset["validation"] = dataset["validation"].shuffle(seed=42).select(range(500))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(500))


Labels: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


In [4]:
# Tokenization & Preprocessing
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocessor(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

# Map preprocessor to dataset
dataset = dataset.map(preprocessor, batched=True)
if "label" in dataset["train"].column_names:
    dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 500/500 [00:00<00:00, 11277.80 examples/s]


In [5]:
# Model Setup
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)
model.to(device)

# Applying LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1169.59it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

trainable params: 299,526 || all params: 109,786,380 || trainable%: 0.2728


In [6]:
# Metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [8]:
# Training Setup
args = TrainingArguments(
    output_dir="./lora_emotion",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    num_train_epochs=2,    
    eval_strategy="epoch",
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics
)

In [9]:
# Train Model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.342855,1.296328,0.538000
2,1.032768,1.132368,0.588000


TrainOutput(global_step=750, training_loss=1.2997066497802734, metrics={'train_runtime': 120.1606, 'train_samples_per_second': 49.933, 'train_steps_per_second': 6.242, 'total_flos': 396060973056000.0, 'train_loss': 1.2997066497802734, 'epoch': 2.0})

In [10]:
# Save the Adapter
model.save_pretrained("./lora_emotion_adapter")
print("LoRA fine-tuning complete! Adapter saved at ./lora_emotion_adapter")

LoRA fine-tuning complete! Adapter saved at ./lora_emotion_adapter


In [11]:
# Test the Model on Custom Sentences
samples = [
    "I am so happy to see you!",
    "This is terrifying, I can't handle it.",
    "He surprised everyone with his gift.",
    "I feel so sad and lonely.",
    "I love spending time with my family."
]

inputs = tokenizer(samples, truncation=True, padding=True, return_tensors="pt").to(device)

model.eval()
with torch.no_grad():
    logits = model(**inputs).logits
    preds = torch.argmax(logits, dim=-1)

for text, pred in zip(samples, preds):
    print(f"Text: {text}\nPredicted Emotion: {emotion_labels[pred]}\n")

Text: I am so happy to see you!
Predicted Emotion: joy

Text: This is terrifying, I can't handle it.
Predicted Emotion: sadness

Text: He surprised everyone with his gift.
Predicted Emotion: joy

Text: I feel so sad and lonely.
Predicted Emotion: sadness

Text: I love spending time with my family.
Predicted Emotion: joy

